# Predicción de Retraso en la Notificación de Dengue — Región Piura

**Proyecto de Investigación en IA — Parte Práctica**

Este notebook es el documento único y definitivo de la parte práctica del proyecto. Contiene, en orden, todo el flujo de trabajo: comprensión de datos, EDA, preprocesamiento, modelado y evaluación. Cada sección de código está etiquetada con la sección correspondiente de la estructura de 29 secciones del proyecto, para que sepas exactamente qué capturas/evidencia usar en cada parte del informe.

**Dataset:** Casos de dengue, Región Piura (Gobierno Regional de Piura) — 47,082 registros reales, línea de casos individuales.

**Problema:** Clasificación binaria — predecir si un caso será notificado con retraso (más de 5 días desde el inicio de síntomas hasta la notificación al sistema de salud).

**Técnicas comparadas:** Regresión Logística vs. Random Forest (+ baseline Dummy obligatorio).

---
**Índice del notebook:**
1. Configuración inicial
2. Comprensión de los datos (Sección 15)
3. Análisis Exploratorio de Datos — EDA (Sección 16.3)
4. Preparación e ingeniería de datos (Secciones 16.4, 16.5)
5. Diseño y entrenamiento de modelos (Secciones 17, 18)
6. Evaluación y comparación de modelos (Sección 20)
7. Ajuste del umbral de decisión (Sección 19 - Diseño de experimentación)
8. Conclusión de la parte práctica


## 1. Configuración inicial

Todas las librerías se importan **una sola vez aquí**, así no hace falta repetirlas en el resto del notebook.

**Antes de ejecutar:** sube `dataset_dengue_actualizado.csv` a este entorno de Colab (ícono de carpeta en el panel izquierdo → ícono de subir archivo).

In [ ]:
# Librerías generales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Librerías de modelado y evaluación (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve,
                              precision_recall_curve, f1_score, precision_score,
                              recall_score, accuracy_score)

# Configuración de gráficos
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", None)

print("Librerías cargadas correctamente.")

In [ ]:
# Carga del dataset crudo (tal como se descargó de la fuente)
df_raw = pd.read_csv("dataset_dengue_actualizado.csv")
print("Registros cargados:", df_raw.shape[0], "| Variables:", df_raw.shape[1])
df_raw.head()

## 2. Comprensión de los datos
### -> Corresponde a la Sección 15 del proyecto (Fase 2 - CRISP-DM)

En este bloque describimos el dataset **crudo, sin modificarlo**, para documentar su procedencia y calidad inicial (ficha técnica de la sección 15).

In [ ]:
# Ficha técnica del dataset: tipos de datos y estructura general
print(df_raw.dtypes)
print("\nRango de FECHA_CORTE:", df_raw['FECHA_CORTE'].min(), "-", df_raw['FECHA_CORTE'].max())

In [ ]:
# Cobertura temporal: ¿hay años con datos completos o faltantes?
print(df_raw['ANO'].value_counts().sort_index())

In [ ]:
# Calidad de datos: valores nulos por columna
print(df_raw.isnull().sum())

In [ ]:
# Cardinalidad de variables categóricas
# (esto decide qué variables son viables para usar como features más adelante)
for c in ['DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SEXO', 'TIPO_EDAD']:
    print(f"{c}: {df_raw[c].nunique()} categorías únicas")

**Interpretación (Ficha técnica - Sección 15):**

| Campo | Valor |
|---|---|
| Fuente | Gobierno Regional de Piura |
| Periodo | 2023, 2025, 2026 (**no hay registros de 2024** — limitación a declarar en Sección 23) |
| Registros | 47,082 |
| Variable objetivo | No viene predefinida — se construye en la Sección 4 de este notebook |
| Formato | CSV |
| Restricción de uso | Datos abiertos institucionales, sin restricción de acceso declarada |

`DISTRITO` tiene 155 categorías (demasiada dispersión para usar como feature directa), mientras que `PROVINCIA` tiene solo 8 — por eso se preferirá `PROVINCIA` en el modelado. No hay nulos relevantes salvo en `LOCALIDAD` (no se usará como feature).

## 3. Análisis Exploratorio de Datos (EDA)
### -> Corresponde a la Sección 16.3 del proyecto (Fase 3 - CRISP-DM)

Aplicamos una limpieza **ligera** (solo para poder explorar correctamente) y calculamos la variable de interés (`DELAY_DIAS`) de forma exploratoria. La limpieza definitiva y la construcción formal del target se hacen en la Sección 4.

In [ ]:
# Limpieza ligera para exploración: normalizar texto y filtrar solo Piura
df = df_raw.copy()
for col in ["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "SEXO", "TIPO_EDAD"]:
    df[col] = df[col].astype(str).str.strip().str.upper()

df = df[df["DEPARTAMENTO"] == "PIURA"].copy()
print("Registros tras filtrar Piura:", df.shape[0])

In [ ]:
# Conversión de fechas (formato YYYYMMDD) y cálculo exploratorio del retraso
for col in ["FECHA_INICIO_SINTOMAS", "FECHA_NOTIFICACION"]:
    df[col + "_DT"] = pd.to_datetime(df[col].astype(str), format="%Y%m%d", errors="coerce")

df["DELAY_DIAS"] = (df["FECHA_NOTIFICACION_DT"] - df["FECHA_INICIO_SINTOMAS_DT"]).dt.days
print(df["DELAY_DIAS"].describe())

In [ ]:
# Gráfico 1: Distribución del retraso en notificación
plt.figure(figsize=(8, 4))
plt.hist(df["DELAY_DIAS"].clip(lower=-10, upper=90), bins=30)
plt.title("Distribución del retraso en notificación (días)")
plt.xlabel("Días")
plt.ylabel("Frecuencia")
plt.show()

**Interpretación (Gráfico 1):** la mayoría de casos se notifica en pocos días (mediana ≈ 2 días), pero existe una cola larga de retrasos extremos. Esto sugiere heterogeneidad importante en la capacidad de respuesta del sistema de salud según el caso.

In [ ]:
# Gráfico 2: Casos por año
df["ANO"].value_counts().sort_index().plot(kind="bar", figsize=(6,4), title="Casos por año")
plt.show()

**Interpretación (Gráfico 2):** 2023 concentra casi el doble de casos que 2025 o 2026, y no hay registros de 2024. Puede reflejar una variación epidemiológica real o un cambio en el sistema de vigilancia — se declara como limitación en la Sección 23.

In [ ]:
# Gráfico 3: Casos por mes de inicio de síntomas (estacionalidad)
df["MES_SINTOMAS"] = df["FECHA_INICIO_SINTOMAS_DT"].dt.month
df["MES_SINTOMAS"].value_counts().sort_index().plot(kind="bar", figsize=(6,4), title="Casos por mes de inicio de síntomas")
plt.show()

**Interpretación (Gráfico 3):** pico claro entre marzo y mayo, coincidiendo con la temporada de lluvias/calor en Piura — periodo de mayor proliferación del mosquito vector (*Aedes aegypti*). Esta estacionalidad se confirmará más adelante como una de las variables más influyentes en ambos modelos (Sección 6).

In [ ]:
# Gráfico 4: Distribución de edad de los pacientes
plt.figure(figsize=(8, 4))
plt.hist(df["EDAD"], bins=30)
plt.title("Distribución de edad de los pacientes")
plt.show()

In [ ]:
# Tabla clave: retraso promedio por provincia
retraso_provincia = df.groupby("PROVINCIA")["DELAY_DIAS"].mean().sort_values(ascending=False)
print(retraso_provincia)

**Interpretación (Tabla - hallazgo más importante del EDA):** **Huancabamba tiene un retraso promedio de ~19.8 días**, muy por encima de Piura (4.8) y Sullana (2.6). Esta es evidencia cuantitativa directa de una brecha geográfica de acceso a salud, y se usa como argumento central en la Sección 5 (Situación problemática) y Sección 9 (Justificación) del informe.

## 4. Preparación e ingeniería de datos
### -> Corresponde a las Secciones 16.4 (Preparación) y 16.5 (Prevención de data leakage)

Aquí se hace la limpieza **definitiva**, se decide el umbral de la variable objetivo, se seleccionan las features finales y se codifican.

In [ ]:
# Limpieza de inconsistencias: retrasos negativos (errores de registro) y outliers extremos
n_negativos = (df["DELAY_DIAS"] < 0).sum()
print(f"Casos con retraso negativo (se eliminan, son errores de registro): {n_negativos}")
df = df[df["DELAY_DIAS"] >= 0].copy()

print(df["DELAY_DIAS"].quantile([0.90, 0.95, 0.99, 1.0]))
df = df[df["DELAY_DIAS"] <= 90].copy()  # elimina outliers extremos poco representativos
print("Registros finales tras limpieza:", df.shape[0])

In [ ]:
# Selección del umbral para la variable objetivo: se prueban varios y se elige el más argumentable
for umbral in [3, 5, 7, 10, 14]:
    proporcion = (df["DELAY_DIAS"] > umbral).mean()
    print(f"Umbral {umbral} días -> % de casos 'tardíos': {proporcion:.2%}")

**Justificación del umbral elegido (5 días):** la mediana de retraso es 2 días — es decir, la mitad de los casos se notifica casi de inmediato. Un corte en 5 días captura retrasos **más del doble** de lo típico, reflejando una demora real y no ruido estadístico. Además deja ~16.7% de casos en la clase minoritaria — manejable sin necesitar técnicas avanzadas de balanceo (SMOTE), solo `class_weight="balanced"` (ver Sección 5).

In [ ]:
# Construcción definitiva de la variable objetivo
UMBRAL_DIAS = 5
df["RETRASO_NOTIFICACION"] = (df["DELAY_DIAS"] > UMBRAL_DIAS).astype(int)
print(df["RETRASO_NOTIFICACION"].value_counts(normalize=True))

In [ ]:
# Selección final de features y codificación (one-hot)
# EDAD: numérica | SEXO, PROVINCIA, MES_SINTOMAS: categóricas de baja cardinalidad
# Se excluye DISTRITO (155 categorías, demasiada dispersión)
# Se excluye TIPO_EDAD (98.9% un solo valor, sin varianza informativa)

features_num = ["EDAD"]
features_cat = ["SEXO", "PROVINCIA", "MES_SINTOMAS"]

df_model = df[features_num + features_cat + ["RETRASO_NOTIFICACION"]].dropna().copy()
df_model = df_model[df_model["SEXO"].isin(["MASCULINO", "FEMENINO"])]  # descarta valores atípicos de sexo

df_model = pd.get_dummies(df_model, columns=features_cat, drop_first=True)

X = df_model.drop(columns=["RETRASO_NOTIFICACION"])
y = df_model["RETRASO_NOTIFICACION"]
print("Shape final de X:", X.shape)
print("Columnas:", list(X.columns))

**Prevención de data leakage (obligatorio - Sección 16.5):** todas las variables usadas (`EDAD`, `SEXO`, `PROVINCIA`, `MES_SINTOMAS`) están disponibles **en el momento en que el paciente llega al centro de salud** — ninguna depende de información posterior al evento de notificación (por ejemplo, no se usa `FECHA_NOTIFICACION` como feature, ya que forma parte del cálculo del propio target). Por lo tanto, no existe fuga de información entre lo que el modelo "sabría" en el momento real de predecir y lo que efectivamente usa.

In [ ]:
# Guardado del dataset procesado (buena práctica de reproducibilidad - Sección 20.7)
df_model.to_csv("dengue_piura_procesado.csv", index=False)
print("Guardado: dengue_piura_procesado.csv")

## 5. Diseño y entrenamiento de modelos
### -> Corresponde a las Secciones 17 (Diseño de la solución) y 18 (Desarrollo e implementación)

**Arquitectura de la solución:** Dataset (CSV) → Limpieza y feature engineering → Codificación → Split train/test estratificado → Modelos (Baseline, Logística, Random Forest) → Evaluación.

**Justificación de las técnicas (Sección 18.1):**
- **Regresión Logística:** modelo interpretable, rápido, bajo riesgo de sobreajuste; los coeficientes se interpretan directamente como efecto sobre la probabilidad de retraso.
- **Random Forest:** captura relaciones no lineales e interacciones entre variables (p. ej. edad x provincia); entrega `feature_importance` sin preprocesamiento adicional.
- Se descartó KNN por ser sensible a la escala y a la alta dimensionalidad generada por las variables categóricas codificadas (one-hot).

In [ ]:
# Split train/test ESTRATIFICADO (mantiene la proporción de clases en ambos conjuntos)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Proporción clase 1 (tardío) en train:", round(y_train.mean(), 3), "| en test:", round(y_test.mean(), 3))

In [ ]:
# Escalado (solo necesario para Regresión Logística; Random Forest no lo requiere)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# BASELINE obligatorio (Sección 18.2): referencia mínima que todo modelo debe superar
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
print("Baseline entrenado (predice siempre la clase mayoritaria).")

In [ ]:
# MODELO 1: Regresión Logística
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train_scaled, y_train)
y_pred_log = logreg.predict(X_test_scaled)
y_proba_log = logreg.predict_proba(X_test_scaled)[:, 1]
print("Regresión Logística entrenada.")

In [ ]:
# MODELO 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
print("Random Forest entrenado.")

## 6. Evaluación y comparación de modelos
### -> Corresponde a la Sección 20 (Evaluación) completa

Esta es la sección con más peso de interpretación en el informe: nunca se reporta solo Accuracy.

In [ ]:
# Función auxiliar para no repetir código de evaluación
def evaluar(nombre, y_true, y_pred, y_proba=None):
    resultado = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }
    resultado["ROC-AUC"] = roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan
    return resultado

resultados = pd.DataFrame([
    evaluar("Baseline (Dummy)", y_test, y_pred_base, None),
    evaluar("Regresión Logística", y_test, y_pred_log, y_proba_log),
    evaluar("Random Forest", y_test, y_pred_rf, y_proba_rf),
])
print(resultados.round(3))

**Interpretación (Tabla comparativa - la "paradoja del accuracy"):** el Baseline tiene el accuracy más alto (83.3%) pero Precision y Recall = 0 — solo predice la clase mayoritaria y es inútil en la práctica. Al usar `class_weight="balanced"`, los modelos reales bajan el accuracy mientras suben el Recall (~65%), que es la métrica relevante para este problema (detectar la mayor cantidad posible de casos tardíos reales).

In [ ]:
# Matrices de confusión de los 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, y_pred) in zip(axes, [("Baseline", y_pred_base), ("Logística", y_pred_log), ("Random Forest", y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No tardío", "Tardío"]).plot(ax=ax, colorbar=False)
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC comparadas
plt.figure(figsize=(6, 5))
for nombre, y_proba in [("Regresión Logística", y_proba_log), ("Random Forest", y_proba_rf)]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{nombre} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Azar")
plt.xlabel("Falsos Positivos")
plt.ylabel("Verdaderos Positivos")
plt.title("Curva ROC comparada")
plt.legend()
plt.show()

**Interpretación (ROC-AUC ~0.65 en ambos modelos):** rendimiento modesto pero significativamente mejor que el azar (0.5). Logística y Random Forest son prácticamente equivalentes (0.645 vs 0.655) — esto indica que la relación entre las variables disponibles y el retraso es en gran parte lineal, y que aumentar la complejidad del modelo no aporta una mejora sustancial. Argumento de parsimonia: se prefiere el modelo más simple (logística) al lograr rendimiento equivalente con mayor interpretabilidad.

In [ ]:
# Interpretabilidad - Regresión Logística (coeficientes)
coef_df = pd.DataFrame({"Variable": X.columns, "Coeficiente": logreg.coef_[0]}).sort_values("Coeficiente", key=abs, ascending=False)
print("Top 10 variables más influyentes (Regresión Logística):")
print(coef_df.head(10))

In [ ]:
# Interpretabilidad - Random Forest (feature importance)
importancia_df = pd.DataFrame({"Variable": X.columns, "Importancia": rf.feature_importances_}).sort_values("Importancia", ascending=False)
print("Top 10 variables más importantes (Random Forest):")
print(importancia_df.head(10))

plt.figure(figsize=(8, 5))
top10 = importancia_df.head(10)
plt.barh(top10["Variable"], top10["Importancia"])
plt.gca().invert_yaxis()
plt.title("Importancia de variables - Random Forest")
plt.tight_layout()
plt.show()

**Interpretación (convergencia entre ambos modelos - hallazgo más fuerte del proyecto):** `PROVINCIA_HUANCABAMBA`, `MES_SINTOMAS_5` (mayo) y `EDAD` aparecen como variables top en **ambos** modelos. Esta coincidencia entre dos algoritmos con lógicas de aprendizaje distintas es evidencia de que la señal es real, no ruido del modelo, y conecta directamente con los hallazgos del EDA (Sección 3): Huancabamba como zona de brecha de acceso, y mayo como pico estacional con mayor alerta/respuesta epidemiológica.

## 7. Ajuste del umbral de decisión
### -> Corresponde a la Sección 19 (Diseño de experimentación) y complementa la Sección 20.3

Por defecto, un modelo de clasificación usa 0.5 como corte de probabilidad. Aquí se evalúa si otro punto de corte mejora el balance Precision/Recall según el objetivo del problema (priorizar la detección temprana de casos tardíos).

In [ ]:
# Prueba de distintos umbrales de decisión (usando las probabilidades de Random Forest)
for umbral in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_umbral = (y_proba_rf >= umbral).astype(int)
    p = precision_score(y_test, y_pred_umbral, zero_division=0)
    r = recall_score(y_test, y_pred_umbral, zero_division=0)
    f1 = f1_score(y_test, y_pred_umbral, zero_division=0)
    print(f"Umbral {umbral} -> Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f}")

In [ ]:
# Curva Precision-Recall completa (visualiza el trade-off)
precisions, recalls, umbrales = precision_recall_curve(y_test, y_proba_rf)
plt.figure(figsize=(6, 5))
plt.plot(umbrales, precisions[:-1], label="Precision")
plt.plot(umbrales, recalls[:-1], label="Recall")
plt.xlabel("Umbral de decisión")
plt.ylabel("Valor")
plt.title("Trade-off Precision-Recall según el umbral")
plt.legend()
plt.show()

**Justificación del umbral final elegido (0.4):** no es el matemáticamente óptimo en F1 (ese sería 0.5, con F1=0.344), pero se elige por el **costo asimétrico de los errores** en un sistema de alerta epidemiológica: un falso negativo (no detectar un caso que sí se notificará tarde, y por tanto no intervenir a tiempo) es más costoso que un falso positivo (revisar un caso que resultó no ser tan urgente). Con el umbral 0.4 se gana Recall (83.5% vs 65.2%) perdiendo muy poco F1 (0.333 vs 0.344).

## 8. Conclusión de la parte práctica

| Elemento | Resultado |
|---|---|
| Dataset | Casos de dengue, Región Piura — 47,082 registros reales (Gobierno Regional de Piura) |
| Problema | Clasificación binaria: retraso en notificación (> 5 días) |
| Técnicas comparadas | Regresión Logística vs. Random Forest (+ baseline Dummy) |
| Rendimiento | ROC-AUC ≈ 0.65 en ambos modelos — modesto pero significativamente mejor que el azar |
| Comparación de modelos | Prácticamente equivalentes → se prefiere Regresión Logística por parsimonia/interpretabilidad |
| Hallazgo principal | Huancabamba y la estacionalidad de mayo son predictores consistentes tanto en el EDA como en ambos modelos |
| Umbral de decisión final | 0.4 (prioriza Recall por el costo asimétrico de los errores en salud pública) |
| Limitación principal | Ausencia de variables clínicas/socioeconómicas y vacío de datos en 2024 |

**Veredicto de viabilidad:** el dataset y el enfoque son viables para el proyecto. El rendimiento modesto (AUC ~0.65) es una limitación honesta y reportable, no un defecto del diseño — es esperable en datos administrativos de salud pública con pocas variables disponibles.

**Próximo paso:** trasladar estas tablas, gráficos e interpretaciones a las secciones correspondientes del informe (ver mapeo completo en `guia_notebooks_proyecto.md`).